# Retrieval Architectures: Chroma vs. Typesense

This notebook explores the different retrieval backends supported by the framework. It demonstrates the **retrieval phase**—the process of finding relevant document segments based on a semantic query.

## 1. Local Retrieval with Chroma

Chroma serves as a high-performance local vector database. Retrieval is performed by generating a query embedding and executing a similarity search against the local index.

In [ ]:
import sys
sys.path.append('..')

from src.vectorstores import ChromaVectorStore
from src.embeddings import EmbeddingManager
from src.retrieval import ChromaRAGRetriever

# Initialize backend components
embed_manager = EmbeddingManager()
embed_manager.load_model()
vector_store = ChromaVectorStore()
retriever = ChromaRAGRetriever(vector_store, embed_manager)

# Execute retrieval
query = "Interviewer skills"
results = retriever.retrieve(query, top_k=2)

print(f"--- RETRIEVAL RESULTS (CHROMA) ---\n")
for i, res in enumerate(results):
    print(f"Result {i+1} (Score: {res['similarity_score']:.4f}):")
    print(f"{res['content'][:200]}...\n")

## 2. Remote Retrieval with Typesense

Typesense provides a managed, remote-first vector search experience. The framework integrates with Typesense via LangChain, utilizing remote collections for scalable indexing.

In [ ]:
from src.vectorstores import TypesenseVectorStore
from src.retrieval import TypesenseRAGRetriever
from src.config import AppConfig

cfg = AppConfig()
if not cfg.typesense.api_key:
    print("Typesense API key not configured. Skipping remote retrieval demonstration.")
else:
    ts_store = TypesenseVectorStore()
    ts_retriever = TypesenseRAGRetriever(ts_store)
    
    query = "Interviewer skills"
    
    # Execute remote retrieval
    results = ts_retriever.retrieve(query, top_k=2)
    
    print(f"--- RETRIEVAL RESULTS (TYPESENSE) ---\n")
    for i, res in enumerate(results):
        # metadata contains source info
        source = res['metadata'].get('source_file', 'unknown')
        print(f"Result {i+1} [Source: {source}]:")
        print(f"{res['content'][:200]}...\n")

## Summary

Note that this notebook demonstrates **Retrieval only** (to get context) (finding the relevant text chunks). To see how these chunks are transformed into a human-readable answer by an LLM, please proceed to **`03_LLM_Generation.ipynb`**.